In [1]:

import torch 
from base import model 
from base import tokenizer
import copy 
import sys
import re


policy = model
reference = copy.deepcopy(model)

reference.eval()
for p in reference.parameters():
    p.requires_grad_(False)

PROMPT = "The capital of France is"
TARGET = "Paris"
G, STEPS, MAX_NEW, EPS, BETA, LR,K = 8, 100, 8, 0.2, 0.05, 1e-6, 2
opt = torch.optim.AdamW(policy.parameters(), lr=LR)


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [2]:

def reward(text):
    words = re.findall(r"[a-z]+", text.lower())
    return float(sum(w == TARGET.lower() for w in words))
        
def token_logprobs(model, ids, prompt_len):
    logits = model(ids).logits[:, :-1].float()
    lp = torch.log_softmax(logits, -1).gather(-1, ids[:, 1:, None]).squeeze(-1)
    return lp[:, prompt_len - 1:] # log prob of each generated token 


In [3]:
device = next(policy.parameters()).device
for step in range(STEPS):

    prompt_ids = tokenizer(PROMPT, return_tensors='pt').input_ids.repeat(G, 1)
    prompt_ids = prompt_ids.to(device)
    
    # Policy-Loss
    with torch.no_grad():
        ids = policy.generate(
            prompt_ids,
            max_new_tokens=MAX_NEW,
            do_sample=True,
            top_k=0,
            pad_token_id=tokenizer.eos_token_id
        )
        prompt_length = prompt_ids.shape[-1]
        mask = (ids[:, prompt_length:] == tokenizer.eos_token_id).cumsum(-1) <= 1
        ref_lp = token_logprobs(reference, ids, prompt_length)
        old_lp = token_logprobs(policy, ids, prompt_length) # prev policy, but for each step this is also getting updated so why do we need importance sampling here ?  

    texts = tokenizer.batch_decode(ids[:, prompt_ids.shape[1]:])
    R = torch.tensor([reward(t) for t in texts], device=device)
    A = (R-R.mean()) # baselining
    
    for _ in range(K):
        new_lp = token_logprobs(policy, ids, prompt_length)
        ratio = torch.exp(new_lp - old_lp)
        clipped = torch.clamp(ratio , 1-EPS, 1+EPS)
        pg = torch.min(ratio * A[:, None] , A[:,None] * clipped)
        
        # KLD-Loss
        # kl = (new_lp - ref_lp) ( faulty one)
        kl = torch.exp(ref_lp - new_lp) - (ref_lp - new_lp) - 1
        
        # loss 
        loss = -(pg - BETA * kl)
        loss = (loss * mask).sum() / mask.sum()
        
        opt.zero_grad()
        loss.backward()

        # gradient norm
        total_norm = 0.0
        for p in policy.parameters():
            if p.grad is not None:
                param_norm = p.grad.data.norm(2)
                total_norm += param_norm.item() ** 2

        total_norm = total_norm**0.5
        print(f"Total Gradient Norm: {total_norm}")
        
        torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
        opt.step()

    if step % 10 == 0:
        print(f"step {step:3d}  mean_R={R.mean():.2f}  sample={texts}")

[transformers] The attention mask is not set with a batched input, and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Total Gradient Norm: 5.878156055224415
Total Gradient Norm: 5.877778645349546
step   0  mean_R=0.38  sample=[' the capital of the 43rd', ' the filbert fransis, or', ' the capital of the country and it is', ' the French capital of Paris, the largest', ' Paris, where most people are concentrated around', ' Paris, the political capital of France is', ' Le Havre. La Rochef', ' your financial foundation, and a smooth transition']
Total Gradient Norm: 5.2739866059133975
Total Gradient Norm: 5.342880513257055
Total Gradient Norm: 7.787636196819162
Total Gradient Norm: 7.6683834894607665
Total Gradient Norm: 11.602252192178058
Total Gradient Norm: 11.703639911686725
Total Gradient Norm: 9.23680566048132
Total Gradient Norm: 9.100479194120712
Total Gradient Norm: 4.84619125404646
Total Gradient Norm: 4.812432581255917
Total Gradient Norm: 7.386944672041087
Total Gradient Norm: 7.372541118732239
Total Gradient Norm: 6.880070825502629
Total Gradient Norm: 6.739528087308956
Total Gradient Norm: 8.

### Faulty KL giving this : 
1. The KL term is wrong as a loss, here the gap part is getting erased

```python
l = - (pg - beta * KL)
dl = beta * d(KL)
KL = new_pg - old_pg

where old_pg is frozen, grads = False

d(KL) = d(new_pg)

so new_pg is just a number, a negative number always
d(KL) = -ve 

θ ← θ − lr·β·∇new_lp
```

[transformers] The attention mask is not set with a batched input, and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Total Gradient Norm: 5.064845642579052
Total Gradient Norm: 1.5920419647653878
step   0  mean_R=0.62  sample=[' Paris.\nCapital of France.     ', ' the historic center of Paris, surrounded by', ' the heart of the city. The city', ' Paris. Paris is a major city known', ' the Palace of Versailles, maintained by King', ' the largest city, known for its rich', ' Paris. Paris is a major city in', ' called Paris, which is a cultural and']
Total Gradient Norm: 0.3445889661685117
Total Gradient Norm: 0.38419457429809967
Total Gradient Norm: 0.4168613701708364
Total Gradient Norm: 0.42416443101869017
Total Gradient Norm: 0.3689669689304669
Total Gradient Norm: 0.38980457749954983
Total Gradient Norm: 0.36950607610056657
Total Gradient Norm: 0.38123613819929547
Total Gradient Norm: 0.4877057983863225
Total Gradient Norm: 0.5029923226995655
Total Gradient Norm: 0.5346297148623786
Total Gradient Norm: 0.6210440992206986
Total Gradient Norm: 0.5194086489467397
Total Gradient Norm: 0.5195295409572427
Total Gradient Norm: 0.470437648768961
Total Gradient Norm: 0.4674380316063797
Total Gradient Norm: 0.7676529085029604
Total Gradient Norm: 2.3550479991151487
Total Gradient Norm: 0.49084770574806397
Total Gradient Norm: 0.48692875228662397
step  10  mean_R=0.00  sample=[' France.\nQ: In what city', ' France and science is science. A referendum', ' "France".<|im_end|><|im_end|><|im_end|><|im_end|><|im_end|>', ' "Franconia".\n-', ' France.<|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|>', ' France. Thus, we have F ->', " France'.<|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|>", ' France.<|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|>']
Total Gradient Norm: 0.5450096439960959
Total Gradient Norm: 0.6075239810572153
Total Gradient Norm: 0.568297900852267
Total Gradient Norm: 0.6272805879869998
Total Gradient Norm: 0.5218628479816411
Total Gradient Norm: 0.5599008592067671
Total Gradient Norm: 0.4076597155952474
Total Gradient Norm: 0.42116708601254604
Total Gradient Norm: 0.4918283804403321
Total Gradient Norm: 0.5086331194865785
Total Gradient Norm: 0.42323012405487254
Total Gradient Norm: 0.4077885100241665
Total Gradient Norm: 0.5194683377457576
Total Gradient Norm: 0.5198240308906378
Total Gradient Norm: 0.43104420358256657
Total Gradient Norm: 0.4357661964123533
Total Gradient Norm: 0.4007038325560147
Total Gradient Norm: 0.40049463591514045
Total Gradient Norm: 0.5095986802223682
Total Gradient Norm: 0.508743621653524
step  20  mean_R=0.00  sample=[' "(fr)" denotes capital of France,', ' n) The value of γ = ', ' écharder. Which is the', ' "Véhicule historique",', ' 3.\n\nSubstituting the', ' A, and that in its non-', ' UTC+9.\n\n### D', ' French\nCapitalizing foreign language phrases\n']
Total Gradient Norm: 0.49715188266781535
Total Gradient Norm: 0.5079076636535059
Total Gradient Norm: 0.4956250415305492
Total Gradient Norm: 0.49409897585389384
Total Gradient Norm: 0.4527248903154062
Total Gradient Norm: 0.46381191965928215
Total Gradient Norm: 0.4583252023635837
Total Gradient Norm: 0.4625224601460693
Total Gradient Norm: 2.4667379628212083
Total Gradient Norm: 1.9994866640899
Total Gradient Norm: 0.4924662455053302
Total Gradient Norm: 0.5070133482141503
Total Gradient Norm: 0.4375525623517007
Total Gradient Norm: 0.4392966034750538
Total Gradient Norm: 0.4984686828901428
Total Gradient Norm: 0.506031343199265
Total Gradient Norm: 0.521312013637534
Total Gradient Norm: 0.5276240334670697
Total Gradient Norm: 0.5183253132310955
Total Gradient Norm: 0.5380341244522252
step  30  mean_R=0.00  sample=[' N.<|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|>', ' given by a French vowel.\nWe', ' 86,000 euros', ' 1000, thereby giving', ' not Equivalent for English, but rather', " 'france s'À la", ' benét, so La Répub', ' Napier . The actual pirate Fort Marl']
Total Gradient Norm: 0.5310489378369206
Total Gradient Norm: 0.5405182187779871
Total Gradient Norm: 0.515154075386655
Total Gradient Norm: 0.5251390183067475
Total Gradient Norm: 0.511302199359425
Total Gradient Norm: 0.5441418675521786
Total Gradient Norm: 0.4867276873832845
Total Gradient Norm: 0.4912070780544644
Total Gradient Norm: 0.6230777610250459
Total Gradient Norm: 0.6380618416329454
Total Gradient Norm: 0.4643876323957814
Total Gradient Norm: 0.47283134789738335
Total Gradient Norm: 0.41701354621463377
Total Gradient Norm: 0.417001996396789
Total Gradient Norm: 0.4232088110183568
Total Gradient Norm: 0.42231216238458885
Total Gradient Norm: 0.5009165661934573
Total Gradient Norm: 0.5168594091960408
Total Gradient Norm: 0.4879670249047401
Total Gradient Norm: 0.48816182869614316
step  40  mean_R=0.00  sample=[' a cuboid (has a base area', ' a perfect square and thus less than ', ' J = T * U / W^', ' squaring trinomial ((π^', ' ${}{\\Set}_F {{\\', ' dividing the first letter of Luxembourg and the', ' |x| + |y| +', ' 10*5=50']
Total Gradient Norm: 0.4286122263429106
Total Gradient Norm: 0.43127476834979867
Total Gradient Norm: 0.4794653290890679
Total Gradient Norm: 0.4851216378094429
Total Gradient Norm: 0.40785618876079555
Total Gradient Norm: 0.4106197276687113
Total Gradient Norm: 0.4010463913148596
Total Gradient Norm: 0.4083090528082785
Total Gradient Norm: 0.3949937000989885
Total Gradient Norm: 0.3991746720817363
Total Gradient Norm: 0.43801411740704904
Total Gradient Norm: 0.4421316328757498
Total Gradient Norm: 0.3836798805509487
Total Gradient Norm: 0.3857479741056309
Total Gradient Norm: 0.45188721864621606
Total Gradient Norm: 0.45637853512046855
Total Gradient Norm: 0.41946876421998597
Total Gradient Norm: 0.4202486691285106
Total Gradient Norm: 0.46663717718896763
Total Gradient Norm: 0.46865085388985306
step  50  mean_R=0.00  sample=[' 2p^2 = 2', ' zero for any two natural numbers a and', ' indeed pi squared minus pi squared plus ', ' called absolute zà0€ bz', ' approximately 11.25\n', ' (b1 * b2) +', ' denoted as $f({^{-1', ' gotten directly as the sum of consecutive integers']
Total Gradient Norm: 0.4964681920259597
Total Gradient Norm: 0.5089835833417455
Total Gradient Norm: 0.5316208071679702
Total Gradient Norm: 0.5408965356917249
Total Gradient Norm: 0.46446962188472174
Total Gradient Norm: 0.46162883925564846

### Using correct KL we get : grpo loss 
kl = torch.exp(ref_lp - new_lp) - (ref_lp - new_lp) - 1




[transformers] The attention mask is not set with a batched input, and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Total Gradient Norm: 6.079212799892945
Total Gradient Norm: 5.946313161013332
step   0  mean_R=0.38  sample=[' Paris.\nParis is situated at the', ' Leîle de la Chapelle (', ' Paris, and the Louvre Museum in', ' the City of light streets.\nCapital', ' the Fédération Anonyme', ' the largest city in the country, with', ' Paris, the bustling city off the Se', ' located in the Alsace region.']
Total Gradient Norm: 5.174904996004502
Total Gradient Norm: 5.103445215392825
Total Gradient Norm: 4.947201339478708
Total Gradient Norm: 5.113906520500946
Total Gradient Norm: 4.774888478734992
Total Gradient Norm: 4.664398287102646
Total Gradient Norm: 4.9648986328657125
Total Gradient Norm: 5.103047941832636
Total Gradient Norm: 4.008331319623156
Total Gradient Norm: 3.981666965784835
Total Gradient Norm: 6.510539064146893
Total Gradient Norm: 6.514134328001914
Total Gradient Norm: 7.147585738878011
Total Gradient Norm: 6.793631555863586
Total Gradient Norm: 5.158797399223627
Total Gradient Norm: 5.248039295289018
Total Gradient Norm: 5.274183059899925
Total Gradient Norm: 5.122719106322014
Total Gradient Norm: 6.055816374107319
Total Gradient Norm: 6.054173356653295
step  10  mean_R=0.50  sample=[' Paris, a city known for its iconic', ' the fédérat, a city', ' the City of Light, a unique ecclesiastical', ' located in the northern part of the country', '  350,000', ' Paris, divided into 12 arr', ' Paris, which is also known as the', ' Paris, a grand historic city covering over']
Total Gradient Norm: 5.32592358375893
Total Gradient Norm: 5.108564524356231
Total Gradient Norm: 5.667242670119335
Total Gradient Norm: 5.909018590468331
Total Gradient Norm: 6.243614131625297
Total Gradient Norm: 6.400597163602577
Total Gradient Norm: 5.772533925794895
Total Gradient Norm: 5.7094278195975345
Total Gradient Norm: 5.575381183850133
Total Gradient Norm: 5.643736164893495
Total Gradient Norm: 6.594131797656717
Total Gradient Norm: 6.841188151607793
Total Gradient Norm: 4.890254497204817
Total Gradient Norm: 4.809282448766081
Total Gradient Norm: 6.198251452874049
Total Gradient Norm: 6.3477415859648465
Total Gradient Norm: 3.0126014269786308
Total Gradient Norm: 2.9150003302858805
Total Gradient Norm: 5.307211310652146
Total Gradient Norm: 5.184985119983139
step  20  mean_R=0.62  sample=[' the capital of the state of France.', ' Paris.\n\nبعد�', ' Paris, with the Latin name of "', ' Paris. Paris is a city that is', ' the City of Light. \n\nForm', ' Paris is the most populous city in the', ' a public works project, known as La', ' Notre Dame de Paris. The city is']
Total Gradient Norm: 7.546850608209717
Total Gradient Norm: 7.421867981957581
Total Gradient Norm: 6.501939421181233
Total Gradient Norm: 6.245803865644641
Total Gradient Norm: 5.856463980760283
Total Gradient Norm: 5.829945154766348
Total Gradient Norm: 5.339313759834914
Total Gradient Norm: 5.368157512902991
Total Gradient Norm: 5.4969917670578505
Total Gradient Norm: 5.5811352077359295
Total Gradient Norm: 5.372307492879854
Total Gradient Norm: 5.494447550047289
Total Gradient Norm: 5.494055961935672
Total Gradient Norm: 5.201873819311417
Total Gradient Norm: 4.62430059454327
Total Gradient Norm: 4.66411337672972
Total Gradient Norm: 7.765010422770536
Total Gradient Norm: 7.658080817298828
Total Gradient Norm: 4.60584492396845
Total Gradient Norm: 4.4936448229497055
step  30  mean_R=0.75  sample=[' Paris.\n3. The famous Picasso', ' the historic city of Paris, a cultural', ' Paris, on the north bank of the', ' Windères.\n\nIt may', ' Paris, France. The area of Paris', ' Paris.\n * Paris is nicknamed the', ' the historic city of Paris. Paris is', ' the Palace of Versailles.\n19']
Total Gradient Norm: 6.493791118194254
Total Gradient Norm: 6.405565524169894
Total Gradient Norm: 4.367180254457636
Total Gradient Norm: 5.3019294099692615
Total Gradient Norm: 5.676986761904967
Total Gradient Norm: 5.6396469063364565
Total Gradient Norm: 6.425853987400954
Total Gradient Norm: 6.506213566470123
Total Gradient Norm: 4.8078867270883014
Total Gradient Norm: 4.990673068298187
Total Gradient Norm: 3.0466453221146947
Total Gradient Norm: 3.085842820417238
Total Gradient Norm: 3.167392537199166
Total Gradient Norm: 3.242370848469617
Total Gradient Norm: 2.995060064911009
Total Gradient Norm: 3.254247139800341
Total Gradient Norm: 5.123437535715524
Total Gradient Norm: 5.202241669383875
Total Gradient Norm: 5.756883524276606
Total Gradient Norm: 5.91530766978599
step  40  mean_R=0.50  sample=[' Paris. It is the main city and', ' the Île de la Cité', ' Paris. Paris is known for its historical', " the Palace of Versailles. It's a", ' the Palace of Versailles. It was built', ' located in the Latin Republic. The capital', ' Paris.\n- Thebes is the central', ' Paris, a magnificent city filled with historic']
Total Gradient Norm: 5.332988003419984
Total Gradient Norm: 5.4454663280941284
Total Gradient Norm: 4.889489584434323
Total Gradient Norm: 4.933349071726739
Total Gradient Norm: 5.746064225567678
Total Gradient Norm: 5.889902976229488
Total Gradient Norm: 6.734612508289638
Total Gradient Norm: 6.726363700826584
Total Gradient Norm: 5.099354827091483
Total Gradient Norm: 5.210143595016745
Total Gradient Norm: 7.681282565658542
Total Gradient Norm: 7.89708065109363
Total Gradient Norm: 6.8011428653606005
Total Gradient Norm: 6.673332320974069
Total Gradient Norm: 5.477739912252124
Total Gradient Norm: 5.474782401381089
Total Gradient Norm: 6.408289419591307
Total Gradient Norm: 6.626634275870255
Total Gradient Norm: 6.221292686371528
Total Gradient Norm: 6.462183076073553
step  50  mean_R=0.38  sample=[" one of the world's most populous cities", ' Île-Saint-Louis (', ' [Capital] Paris, which has a', ' Paris. Paris is a beautiful city with', ' called Paris. Paris is a beautiful city', ' Lauscombe and Dover has as its', ' Le Havre.\n\nLe Hav', ' in the 2nd estate, which']
Total Gradient Norm: 5.200637248427907
Total Gradient Norm: 5.138817005435122
Total Gradient Norm: 5.024691460813865
Total Gradient Norm: 5.087095667848337
Total Gradient Norm: 2.9459370335462918
Total Gradient Norm: 3.0385401308702926
Total Gradient Norm: 5.752812289981319
Total Gradient Norm: 5.095949577141316
Total Gradient Norm: 5.630061469837185
Total Gradient Norm: 5.717839204978428
Total Gradient Norm: 0.0365687621525396
Total Gradient Norm: 0.0218347132921486
Total Gradient Norm: 4.924820676978466
Total Gradient Norm: 4.992189489432881
Total Gradient Norm: 5.433481540933661
Total Gradient Norm: 7.246979960001855
Total Gradient Norm: 5.497218692665671
Total Gradient Norm: 5.871608009836062
Total Gradient Norm: 5.4963569401866845
Total Gradient Norm: 5.419349782151127
step  60  mean_R=0.50  sample=[' Paris, the most populous and scenic capital', ' Paris. Paris is also the political,', ' Paris, known for its historical landmarks like', ' the capital, filled with its great beauty', ' its own unique heritage.\n\nThe', ' Versailles.\nRating: Greek Orthodox', ' located in the northern part of the country', ' Paris, a City of Light, known']
Total Gradient Norm: 3.6834055007008395
Total Gradient Norm: 3.7498032682193503
Total Gradient Norm: 5.562037240091142
Total Gradient Norm: 5.778518224656421
Total Gradient Norm: 4.066424970281423
Total Gradient Norm: 4.008921650542012
Total Gradient Norm: 6.587058555652664
Total Gradient Norm: 6.604059134374039
Total Gradient Norm: 4.889198614708506
Total Gradient Norm: 4.818304611284508
Total Gradient Norm: 8.1299008517595
Total Gradient Norm: 8.1095918247879
Total Gradient Norm: 4.908116080289858
Total Gradient Norm: 4.880189776046046
Total Gradient Norm: 6.879758888141091
Total Gradient Norm: 6.499325846823586
Total Gradient Norm: 5.786414553708137
Total Gradient Norm: 5.918492559347276
Total Gradient Norm: 5.077414702110333
Total Gradient Norm: 5.143109398103256
step  70  mean_R=0.62  sample=[' Les Bâtiments in central Paris', ' capital of... rose citrus  rose pet', ' Paris. Paris is along the Seine', ' Paris, which is located in the heart', ' the city of Paris and is the economic', ' Paris, a city located in the northern', ' the capital of the Principality, which', ' the Seine River, which flows from']
Total Gradient Norm: 4.845765059235623
Total Gradient Norm: 4.893386684160941
Total Gradient Norm: 7.231617469979248
Total Gradient Norm: 7.192008039418014
Total Gradient Norm: 5.268471739277757
Total Gradient Norm: 5.213069334214877
Total Gradient Norm: 5.45281322976166
Total Gradient Norm: 5.507705514288118
Total Gradient Norm: 4.184770159138951
Total Gradient Norm: 4.085222185295469
Total Gradient Norm: 6.046686809344243
Total Gradient Norm: 5.952175034548861
Total Gradient Norm: 4.338737563988039
Total Gradient Norm: 4.328174338162832
Total Gradient Norm: 6.375775124138272
Total Gradient Norm: 6.915376583590062
Total Gradient Norm: 7.09906337900077
Total Gradient Norm: 6.8730069912975225
Total Gradient Norm: 5.457588691611095
Total Gradient Norm: 5.3769444220652955
step  80  mean_R=0.62  sample=[' Paris, which is a cultural, historical', ' the Palace of Versailles. Critics argue that', ' Paris. Paris is a vibrant city known', ' the capital of only the six countries it', ' the Seine River, which is home', ' Paris. Paris is located on the �', " Paris, often referred to as the '", ' Paris.\nNo, London would be']
Total Gradient Norm: 6.100836009687083
Total Gradient Norm: 8.274386069697169
Total Gradient Norm: 6.561318468837465
Total Gradient Norm: 6.126051381920791
Total Gradient Norm: 6.860470247701399
Total Gradient Norm: 6.598500218681629
Total Gradient Norm: 5.77110071343208
Total Gradient Norm: 5.6097605554161865
Total Gradient Norm: 4.910698933530937
Total Gradient Norm: 4.9551241556419585
Total Gradient Norm: 5.18875828528656
Total Gradient Norm: 5.117797548752414
Total Gradient Norm: 6.019660024979774
Total Gradient Norm: 5.909999396542778
Total Gradient Norm: 4.982294403637762
Total Gradient Norm: 4.335143196105552
Total Gradient Norm: 6.768222509534751
Total Gradient Norm: 6.0821145467866895
Total Gradient Norm: 6.152815988975496
Total Gradient Norm: 6.34449617573222
step  90  mean_R=0.62  sample=[' Paris. Paris is the heart of modern', ' the Palace of Versailles).\n\nWhile', ' the Palace of Versailles. The Palace features', ' Paris. Paris is a major cultural,', ' Paris, the largest and most populous urban', " (42° 42'", ' Paris.\n* The capital of Switzerland', ' Paris, a city of classical architectural elements']
Total Gradient Norm: 6.5151659737500305
Total Gradient Norm: 6.543842319986805
Total Gradient Norm: 5.609614631786163
Total Gradient Norm: 5.629013054222938
Total Gradient Norm: 6.379600320118069
Total Gradient Norm: 6.314454151534597
Total Gradient Norm: 5.5686451608784004
Total Gradient Norm: 5.680205330730432
Total Gradient Norm: 4.163886818033655
Total Gradient Norm: 4.36483938875146
Total Gradient Norm: 4.129558189209968
Total Gradient Norm: 4.138445921828612
Total Gradient Norm: 4.899398339869618
Total Gradient Norm: 5.086932388246291
Total Gradient Norm: 6.3913534280916116
Total Gradient Norm: 6.112629875849148
Total Gradient Norm: 6.626685783869272
Total Gradient Norm: 6.684588919502928